<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/01-structured-output.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 结构化输出

**目标：** 从模型那里拿到可靠的 JSON，并看清它到底在哪里、以何种方式出问题。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。


## 准备工作

每个 notebook 都是自包含的，所以接下来两个单元格会从零把它搭起来：

1. **安装依赖。** `aien` 包（本仓库）带着共享的 setup 辅助函数，并拉入 `groq` 客户端——这是本 notebook 唯一需要的依赖。
2. **加载你的 API key。** 到 [console.groq.com](https://console.groq.com/) 领免费 key（无需信用卡）。在 Colab 里，通过左侧边栏的**钥匙图标** → **Add new secret** 添加，名称严格写成 `GROQ_API_KEY`，粘贴其值，并打开 **Notebook access** 开关。在本地运行？把 `GROQ_API_KEY` 设为环境变量。

（完整讲解与选模型指引见 [00-setup/00-environment.ipynb](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/00-setup/00-environment.zh.ipynb)。）

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
from aien import setup

# 读取 GROQ_API_KEY（Colab Secrets 或本地环境变量），返回一个就绪的
# Groq 客户端。传 model=... 可覆盖默认值；如果之后某次调用报 404，
# 就列出可用模型——见 00-setup/00-environment.ipynb。
client, MODEL = setup()

## 问题所在

**本 notebook 的场景：** 你正在构建一个支持系统的受理环节。客户邮件以自由文本形式到达；下游服务需要把每一封都变成一个可路由、可存储的结构化工单，形状固定，像这样：

```json
{
  "summary": "...",
  "account": "...",
  "plan": "enterprise",
  "priority": "urgent"
}
```

那个下游服务不读散文。它调用 `json.loads`、索引 `ticket["priority"]`、插入一行。所以模型的任务不是*描述*邮件，而是每次都精确地吐出那个对象。

这个落差无处不在：任何要喂给数据库、API 调用或另一个程序的 LLM 功能都会撞上它。而那个天真的修法（「直接要 JSON」）会以三种反复出现的方式失败：

- **Markdown 围栏。** 模型把输出包进 ` ```json ... ``` `，`json.loads` 就卡住了。
- **多余的旁白。** 前面来一句「Here's the JSON you asked for:」，后面来一句「Let me know if...」。
- **schema 漂移。** 字段名被改写、枚举值被换词（`"urgent"` → `"P1"`）、数字以字符串返回。

我们会围绕同一个贯穿示例（从下面的客户邮件里抽取一张支持工单），沿着一条越来越稳健的修法阶梯往上爬，从那个天真的调用开始，让你亲眼看到它出问题。

## 阶梯一览

四种方法，稳健程度和投入大致递增。下面每一级都会构建一种，并展示它在哪里出问题；先看这张地图：

| # | 方法 | 能保证什么 | 优点 | 缺点 | 什么时候用它 |
|---|---|---|---|---|---|
| 0 | **直接要 JSON** | 什么都保证不了 | 一行代码，零配置 | 围栏、旁白、漂移；在生产里静默失败 | 生产环境永远不用；它是我们故意用来撞破的基线 |
| 1 | **死磕 prompt + 防御式解析** | 无（尽力而为） | 便宜、可移植、不需要 API 特性 | 仍然信任模型；不检查内容 | 临时脚本；作为下面各级的*降级层* |
| 2 | **JSON mode**（`response_format`） | 合法 JSON（解析器层面） | 消灭围栏/旁白；一个开关 | 不约束*形状*：字段名和枚举仍会漂移 | 整个 schema 都由你在 prompt 里掌控，只想要廉价的解析安全 |
| 3 | **Tool / function calling** | 符合 **schema** 的合法 JSON | schema 是有版本管理的代码；参数到达时已解析；能与真实工具组合 | 配置更多；各家形状不同 | 生产级抽取；这是默认答案 |
| + | **校验 + 重试**（叠加在第 3 级上） | 你的**业务规则** | 能抓到 schema 抓不到的东西（跨字段、数据库存在性） | 失败时多一次调用；需要一个校验器 | 取值必须满足 schema 无法表达的逻辑 |

- **这些是叠加的，不是互相竞争。** 生产级抽取通常是第 3 级 + 校验，再配一个第 1 级的解析作为廉价的安全网。

> **⭐ 关键要点 ——** *解析器*层面的保证不等于 *schema* 层面的保证。JSON mode 承诺这些字节能解析；它完全不保证 `priority` 是否落在你允许的四个值里。把这条区分内化，就是这里的大部分功夫。

现在开始爬，从第 0 级起步，让你亲眼看着它失败。

In [ ]:
# 我们的贯穿示例：一封自由文本的客户邮件，下面每一级都用它。
EMAIL = (
    'Hi team — our checkout page has been throwing 500s since about 2pm. '
    'This is blocking all purchases, we are losing money every minute. '
    'Account: acme-corp, plan: enterprise. Please treat as urgent. — Dana'
)

# 天真的做法：直接要 JSON，祈祷模型乖一点。
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket from this email as JSON with fields '
                   f'summary, account, plan, priority (low/medium/high/urgent):\n\n{EMAIL}',
    }],
)
raw = response.choices[0].message.content
print(raw)

多跑几次。有些运行给出干净的 JSON；有些则把它裹进围栏，或加上一句开场白。

> **🚩 常见错误 ——** 「大多数时候能跑通」是最糟的失败模式：它能在你的演示里活下来，然后在凌晨两点在生产里死掉。把*任何*格式漂移都当成 bug，而不是罕见的烦人小事。

## 第 1 级：死磕 prompt + 防御式解析

收紧 prompt（「只输出 JSON 对象，不要 markdown，不要旁白」），并把解析包在能剥掉常见外壳的错误处理里。这很便宜，能接住大部分问题。即使你用了下面更强的技术，也把这样一个辅助函数留在手边，因为它不花什么成本。


In [ ]:
import json
import re


def parse_json_loose(text):
    """带两种最常见失败模式处理的 json.loads。"""
    text = text.strip()
    # 如果存在 markdown 围栏就剥掉
    fence = re.match(r'^```(?:json)?\s*(.*?)\s*```$', text, re.DOTALL)
    if fence:
        text = fence.group(1)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # 最后手段：抓取最外层的 {...} 区间（丢掉前后的旁白）
        start, end = text.find('{'), text.rfind('}')
        if start != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise


response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket from this email. Output ONLY a JSON object — '
                   f'no markdown fences, no commentary. Fields: summary (string), '
                   f'account (string), plan (string), priority (one of: low, medium, high, urgent).'
                   f'\n\n{EMAIL}',
    }],
)
ticket = parse_json_loose(response.choices[0].message.content)
print(json.dumps(ticket, indent=2))


好一些了，但你仍在信任模型的格式纪律，而且没有任何东西检查*内容*：`priority: "URGENT!!"` 能正常解析，却会在下游把你的枚举弄坏。

## 第 2 级：JSON mode

许多 OpenAI 兼容的 API（包括 Groq）支持 `response_format={"type": "json_object"}`。这会强制响应为合法 JSON，没有围栏也没有开场白，但不校验*形状*。把它看作解析器层面的保证，而不是 schema 层面的保证。


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    response_format={"type": "json_object"},
    messages=[{
        'role': 'user',
        'content': f'Extract a support ticket as JSON with fields summary, account, '
                   f'plan, priority (low/medium/high/urgent):\n\n{EMAIL}',
    }],
)
ticket = json.loads(response.choices[0].message.content)
print(json.dumps(ticket, indent=2))


JSON mode 保证输出可解析，但模型仍可能在字段名或枚举值上漂移。跑一下，注意形状看起来是对的，然后试着把邮件改成提到「platinum plan」，再跑几次，看枚举如何漂移。

## 第 3 级：把 tool use 当作 schema 的强制手段

这是生产级答案。定义一个参数 schema *就是*你输出 schema 的工具，用 `tool_choice` 强制模型调用它，再从 tool call 上读出参数。有三件事你不用额外费力就能得到：

- 响应到达时就是**已解析好的 dict**，完全不需要字符串解析。
- 你可以用真实的逻辑校验*内容*（枚举成员资格、字段长度）。
- schema 是代码，和消费它的代码一起做版本管理，而不是埋在 prompt 里的一段散文。

模型从不「运行」这个工具；我们纯粹是把 tool calling 这套机制借来当一条带类型的输出通道。


In [ ]:
TICKET_TOOL = {
    'type': 'function',
    'function': {
        'name': 'record_ticket',
        'description': 'Record a structured support ticket extracted from a customer email.',
        'parameters': {
            'type': 'object',
            'properties': {
                'summary':  {'type': 'string', 'description': 'One-line issue summary'},
                'account':  {'type': 'string'},
                'plan':     {'type': 'string', 'enum': ['free', 'pro', 'enterprise']},
                'priority': {'type': 'string', 'enum': ['low', 'medium', 'high', 'urgent']},
            },
            'required': ['summary', 'account', 'plan', 'priority'],
        },
    },
}

response = client.chat.completions.create(
    model=MODEL,
    max_tokens=300,
    tools=[TICKET_TOOL],
    tool_choice={'type': 'function', 'function': {'name': 'record_ticket'}},  # 强制调用
    messages=[{'role': 'user', 'content': f'Extract the support ticket:\n\n{EMAIL}'}],
)

tool_call = response.choices[0].message.tool_calls[0]
ticket = json.loads(tool_call.function.arguments)  # 已经是合法 JSON
print(json.dumps(ticket, indent=2))


## 校验失败就重试

工具 schema 覆盖了 JSON Schema 能表达的东西：类型、必填字段、枚举成员资格。（有些厂商还提供 `strict: true` 标志，在解码时就强制 schema；Groq 各模型的支持参差不齐，所以别把它当成唯一的防线。）schema *无法*表达的业务规则（跨字段约束、必须存在于你数据库里的值、依赖上下文的长度上限）仍然需要你自己的校验。

而当校验失败时，正确的做法通常是重试一次，并且*告诉模型哪里错了*。把错误作为一条 tool result 消息发回去，让它用修正后的值再调用一次工具。

In [ ]:
def validate_ticket(t):
    """返回问题列表；空列表表示合法。"""
    problems = []
    if not isinstance(t.get('summary'), str) or not (5 <= len(t['summary']) <= 120):
        problems.append('summary must be a string of 5-120 characters')
    if t.get('priority') not in ('low', 'medium', 'high', 'urgent'):
        problems.append(f"priority {t.get('priority')!r} not in allowed enum")
    if t.get('plan') not in ('free', 'pro', 'enterprise'):
        problems.append(f"plan {t.get('plan')!r} not in allowed enum")
    return problems


def extract_with_retry(email, max_attempts=3):
    messages = [{'role': 'user', 'content': f'Extract the support ticket:\n\n{email}'}]
    for attempt in range(1, max_attempts + 1):
        response = client.chat.completions.create(
            model=MODEL, max_tokens=300,
            tools=[TICKET_TOOL],
            tool_choice={'type': 'function', 'function': {'name': 'record_ticket'}},
            messages=messages,
        )
        tool_call = response.choices[0].message.tool_calls[0]
        ticket = json.loads(tool_call.function.arguments)
        problems = validate_ticket(ticket)
        if not problems:
            print(f'valid on attempt {attempt}')
            return ticket
        print(f'attempt {attempt} failed validation: {problems}')
        # 把 assistant 消息 + tool result 发回去，让模型能自我修正
        messages.append({'role': 'assistant', 'content': None,
                         'tool_calls': response.choices[0].message.tool_calls})
        messages.append({'role': 'tool',
                         'tool_call_id': tool_call.id,
                         'content': 'Validation failed: ' + '; '.join(problems) +
                                    '. Call the tool again with corrected values.'})
    raise ValueError(f'no valid extraction after {max_attempts} attempts')


ticket = extract_with_retry(EMAIL)
print(json.dumps(ticket, indent=2))


跑一下：它应该在第 1 次尝试就通过。这个重试循环是你的安全网，用来兜住 schema 看不见的规则。

## 要记在脑子里的失败模式

即使有 tool-forcing，在真实环境里也要留意这些：

- **枚举漂移。** 没有强制 tool call 时，`low/medium/high/urgent` 这样的 `priority` 枚举偶尔会回成 `"critical"` 或 `"P1"`。带正确 schema 的 tool calling 帮助很大；仍要在自己的代码里校验枚举成员资格作为兜底。
- **可选字段的幻觉。** 加一个可选的 `phone` 字段，模型有时会凭空编一个，而不是省略它，因为模型倾向于把槽位填满。优先使用必填字段配上显式的类 `null` 哨兵值，或者明确指示「源文本中没有的字段就省略」，然后抽查。
- **数字变成字符串。** 出现 `"amount": "42.50"` 而不是 `42.50`，尤其当源文本里有货币符号时。做类型转换并记日志，而不是信任类型。

JSON mode + tool forcing 并用：当整个 schema 都由你在 prompt 里掌控时，用 JSON mode（`response_format`）；当你想让 schema 成为代码（可版本管理、可标注类型、能与其他工具组合）时，用 tool forcing。下一个 notebook 会深入讲 tool calling 协议。


## 练习

1. 改用 `response_format={"type": "json_object"}`（第 2 级），并把邮件里的 `plan` 枚举改成一个不在列表里的值（例如「platinum plan」）。跑 5 次，数一数枚举发生漂移的频率，再和第 3 级的 tool forcing 对比。
2. 给工具 schema 加一个可选的 `affected_since` 字段（ISO 时间戳）。分别喂给它提到时间和没提到时间的邮件，度量模型凭空编造这个字段的频率。
3. 写一个 `extract_batch(emails)`，对 5 封短邮件组成的列表跑第 3 级的抽取器（顺序循环），把成功结果和校验失败分开收集。
4. 把 `validate_ticket` 换成针对同一个 schema dict 的 `jsonschema.validate()` 调用（`jsonschema` 包随 Colab 自带），并改造重试循环，把它的错误消息回传给模型。
